In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_risk= pd.read_csv('HR_Risk_Analysis.csv')
df_risk.head()

In [ ]:
#Total Financial Exposure: "Critical Risk" Lynchpins by Department'
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_risk[df_risk['Retention_Strategy_Action'] == 'CRITICAL RETENTION RISK'], 
    x='Department', 
    y='Estimated_Replacement_Cost_USD', 
    estimator=sum, 
    palette='Reds_r'
)
plt.title('Total Financial Exposure: "Critical Risk" Lynchpins by Department')
plt.ylabel('Aggregate Replacement Cost ($)')
plt.show()

In [ ]:
#Income vs. Satisfaction
# Filter for Tier 1 Lynchpins only
tier1_df = df_risk[df_risk['Position_Tier'] == 'Tier 1 - Mission Critical']

plt.figure(figsize=(10, 6))
sns.swarmplot(data=tier1_df, x='Retention_Strategy_Action', y='MonthlyIncome', hue='OverTime')
plt.title('Tier 1 Lynchpins: Income vs. Satisfaction (Color by Overtime)')
plt.axhline(tier1_df['MonthlyIncome'].mean(), color='red', linestyle='--', label='Avg Tier 1 Salary')
plt.legend(title='Working Overtime?')
plt.show()

In [ ]:
palette_focus = {
    "CRITICAL RETENTION RISK": "#D62728", 
    "STABLE": "#D3D3D3",                 
    "STAGNATION RISK": "#D9F366",          
    "HIGH BURNOUT ALERT": "#39D6F1"        
}

plt.figure(figsize=(10, 6))


sns.kdeplot(
    data=df_risk, 
    x="YearsSinceLastPromotion", 
    hue="Retention_Strategy_Action",
    palette=palette_focus, 
    fill=True, 
    common_norm=False, 
    alpha=0.6, 
    linewidth=2.5
)

plt.title("Strategic Warning: Critical Risk Peaks After 3-Year Promotion Lag", 
          fontsize=14, pad=20)

plt.xlabel("Years Since Last Promotion", fontsize=12)
plt.ylabel("Density of Talent", fontsize=12)

sns.move_legend(plt.gca(), "upper right", frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# Define Revenue Contribution by Role

revenue_map = {
    'Sales Executive': 2000000, 
    'Research Director': 5000000, 
    'Manager': 3000000,
    'Manufacturing Director': 2500000
}

df_risk['Annual_Revenue_Value'] = df_risk['JobRole'].map(revenue_map).fillna(500000) # Default for others

# Calculate Revenue Leakage (assuming 90 days to fill a Tier 1 role)
df_risk['Revenue_Leakage'] = ((df_risk['Annual_Revenue_Value'] / 365) * 90).round(2)

# Calculate TOTAL EXPOSURE
df_risk['Total_Economic_Impact'] = df_risk['Estimated_Replacement_Cost_USD'] + df_risk['Revenue_Leakage'].round(2)

df_risk.head()

In [ ]:
# Top 5 Mission-Critical Positions by Total Economic Impact

position_risk = df_risk.groupby('JobRole').agg(
    Avg_Economic_Impact=('Total_Economic_Impact', 'mean'),
    Role_Criticality=('Position_Tier', 'first')
).reset_index()

top_5_positions = position_risk.sort_values(by='Avg_Economic_Impact', ascending=False).head(5)

plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

ax = sns.barplot(
    data=top_5_positions, 
    x='Avg_Economic_Impact', 
    y='JobRole', 
    palette='flare'
)

for p in ax.patches:
    width = p.get_width()
    plt.text(
        width + 20000, 
        p.get_y() + p.get_height()/2, 
        f'${width:,.2f}', 
        va='center', 
        fontweight='bold',
        color='#7a0177'
    )

plt.title('Top 5 Mission-Critical Positions by Total Economic Impact', fontsize=16, pad=20)
plt.xlabel('Total Economic Impact per Vacancy (USD)', fontsize=12)
plt.ylabel('Position Title', fontsize=12)
plt.xlim(0, top_5_positions['Avg_Economic_Impact'].max() * 1.3)

plt.tight_layout()
plt.show()